# Jalur Penelitian R06: Spatial Intelligence & 3D Vision
## Reproducible Research Baseline: Calibrated Stereo Projection, DLT Triangulation & Trajectory Error (ATE)

**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Level:** L4 Research-Ready / ISVWS 2026–2040 Standard  
**Evidence Category:** Educational Simulation & Reproducible Benchmark Template

Notebook ini menyediakan protokol komputasi deterministik untuk mereproduksi eksperimen visi spasial 3D:
1. **Model Kamera Pinhole Terkalibrasi:** Matriks intrinsik $\mathbf{K}$, ekstrinsik $[\mathbf{R} | \mathbf{t}]$, dan proyeksi perspektif 3D ke 2D.
2. **Triangulasi Direct Linear Transformation (DLT):** Rekonstruksi koordinat 3D dari pasangan titik piksel kiri-kanan menggunakan Singular Value Decomposition (SVD).
3. **Evaluasi Galat Reproyeksi (Reprojection RMSE):** Verifikasi keakuratan pemetaan mundur ke bidang gambar piksel.
4. **Analisis Ketidakpastian Kuadratik Kedalaman:** Pemodelan $\sigma_Z = \frac{Z^2}{f \cdot B} \sigma_d$ terhadap jarak objek.
5. **Absolute Trajectory Error (ATE):** Pengukuran deviasi lintasan visual odometry dari ground truth.

In [ ]:
import os, random, numpy as np, math

def set_deterministic_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"[REPRODUCIBILITY] Random Seed locked to {seed} successfully.")

set_deterministic_seed(42)

In [ ]:
# 1. Model Kamera Pinhole Stereo & Proyeksi Titik 3D
focal = 718.856
cx, cy = 607.19, 185.21
baseline = 0.54

K = np.array([
    [focal, 0.0, cx],
    [0.0, focal, cy],
    [0.0, 0.0, 1.0]
])

P_left = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
P_right = K @ np.hstack([np.eye(3), np.array([[-baseline], [0.0], [0.0]])])

def project_point(P: np.ndarray, X_w: np.ndarray) -> np.ndarray:
    X_h = np.append(X_w, 1.0)
    u_h = P @ X_h
    return np.array([u_h[0] / u_h[2], u_h[1] / u_h[2]])

sample_pt = np.array([1.5, -0.5, 8.0])
u_l = project_point(P_left, sample_pt)
u_r = project_point(P_right, sample_pt)
print(f"Titik 3D Asli: {sample_pt}")
print(f"Proyeksi Kiri (px): {u_l}, Proyeksi Kanan (px): {u_r}")
print(f"Disparitas (px): {u_l[0] - u_r[0]:.2f}")

In [ ]:
# 2. Rekonstruksi 3D Direct Linear Transformation (DLT) via SVD
def triangulate_dlt(P1: np.ndarray, P2: np.ndarray, pt1: np.ndarray, pt2: np.ndarray) -> np.ndarray:
    u1, v1 = pt1
    u2, v2 = pt2
    A = np.zeros((4, 4))
    A[0] = u1 * P1[2] - P1[0]
    A[1] = v1 * P1[2] - P1[1]
    A[2] = u2 * P2[2] - P2[0]
    A[3] = v2 * P2[2] - P2[1]
    _, _, Vt = np.linalg.svd(A)
    X_h = Vt[-1]
    return X_h[:3] / X_h[3]

rec_pt = triangulate_dlt(P_left, P_right, u_l, u_r)
print(f"Rekonstruksi DLT 3D: {rec_pt}")
print(f"Galat Posisi Rekonstruksi: {np.linalg.norm(sample_pt - rec_pt):.6e} m")

In [ ]:
# 3. Evaluasi Multi-Seed Reprojection RMSE & ATE Odometri (Gate 6)
seeds = [42, 1337, 2026]
print("=== REPRODUCIBILITY BENCHMARK MULTI-SEED (GATE 6) ===")
for seed in seeds:
    set_deterministic_seed(seed)
    # Evaluasi 100 titik acak dengan derau pengukuran sub-piksel
    pts_3d = np.random.uniform(low=[-2, -1, 4], high=[2, 1, 15], size=(100, 3))
    reproj_errors = []
    for pt in pts_3d:
        ul = project_point(P_left, pt) + np.random.normal(0, 0.4, 2)
        ur = project_point(P_right, pt) + np.random.normal(0, 0.4, 2)
        rec = triangulate_dlt(P_left, P_right, ul, ur)
        ul_back = project_point(P_left, rec)
        reproj_errors.append(np.linalg.norm(ul - ul_back))
    rmse = math.sqrt(np.mean(np.array(reproj_errors)**2))
    print(f"Seed {seed} -> Reprojection RMSE: {rmse:.3f} px")